In [2]:
#-- vectorization. --#
import time
import numpy as np
import pandas as pd

# Simulating a large dataset with 100,000 rows to demonstrate the speed difference
df_large = pd.DataFrame(
    {
        "Grade": np.random.randint(50, 100, size=100000),
        "Absences": np.random.randint(0, 20, size=100000),
    }
)


# THE AMATEUR WAY: Iterating rows using a for-loop (Very slow)
start_time = time.time()
loop_results = []
for index, row in df_large.iterrows():
    if row["Grade"] > 80 and row["Absences"] < 3:
        loop_results.append("Excellent")
    else:
        loop_results.append("Normal")
df_large["Status_Loop"] = loop_results
print(f"For-Loop Execution Time: {time.time() - start_time:.4f} seconds")

# THE PRO WAY: Using Vectorization with np.select (Blazing fast)
start_time = time.time()
conditions = [(df_large["Grade"] > 80) & (df_large["Absences"] < 3)]
choices = ["Excellent"]

df_large["Status_Vector"] = np.select(conditions, choices, default="Normal")
print(f"Vectorized Execution Time: {time.time() - start_time:.4f} seconds")


For-Loop Execution Time: 2.4930 seconds
Vectorized Execution Time: 0.0175 seconds


In [6]:
# Load the raw dataset
df_mem = pd.read_csv("student_grade.csv")

print("--- MEMORY USAGE BEFORE OPTIMIZATION ---")
print(df_mem.info(memory_usage="deep"))

# 1. Downcast numeric columns to smaller integer types
df_mem["age"] = pd.to_numeric(df_mem["age"], downcast="integer")
df_mem["absences"] = pd.to_numeric(df_mem["absences"], downcast="integer")

# 2. Convert repetitive categorical string columns into 'category' type
cat_cols = ["gender", "major", "tutoring", "parental_support", "curriculum_activities"]
for col in cat_cols:
    df_mem[col] = df_mem[col].astype("category")

print("\n--- MEMORY USAGE AFTER OPTIMIZATION ---")
print(df_mem.info(memory_usage="deep"))


--- MEMORY USAGE BEFORE OPTIMIZATION ---
<class 'pandas.DataFrame'>
RangeIndex: 100 entries, 0 to 99
Data columns (total 10 columns):
 #   Column                 Non-Null Count  Dtype
---  ------                 --------------  -----
 0   student_id             100 non-null    str  
 1   first_name             100 non-null    str  
 2   last_name              100 non-null    str  
 3   age                    100 non-null    int64
 4   gender                 100 non-null    str  
 5   major                  100 non-null    str  
 6   absences               100 non-null    int64
 7   tutoring               100 non-null    str  
 8   parental_support       100 non-null    str  
 9   curriculum_activities  87 non-null     str  
dtypes: int64(2), str(8)
memory usage: 44.2 KB
None

--- MEMORY USAGE AFTER OPTIMIZATION ---
<class 'pandas.DataFrame'>
RangeIndex: 100 entries, 0 to 99
Data columns (total 10 columns):
 #   Column                 Non-Null Count  Dtype   
---  ------                

In [7]:
df_expr = pd.read_csv("student_grade.csv")

# Filter 'AI' students with fewer than 5 absences cleanly using .query()
df_ai_good = df_expr.query("major == 'AI' and absences < 5")
print("--- FILTERED DATA WITH .QUERY() ---")
print(df_ai_good[["student_id", "first_name", "major", "absences"]].head(5))


# Create a new 'risk_score' column using .eval() to bypass heavy internal object copies
# Formula: age * 0.5 + absences * 1.2
df_expr.eval("risk_score = (age * 0.5) + (absences * 1.2)", inplace=True)
print("\n--- NEW COLUMN COMPUTATION WITH .EVAL() ---")
print(df_expr[["student_id", "first_name", "age", "absences", "risk_score"]].head(5))


--- FILTERED DATA WITH .QUERY() ---
   student_id first_name major  absences
1        S002      James    AI         2
5        S006      James    AI         1
12       S013      James    AI         4
21       S022       John    AI         1
33       S034        Bob    AI         0

--- NEW COLUMN COMPUTATION WITH .EVAL() ---
  student_id first_name  age  absences  risk_score
0       S001      Alice   22         4        15.8
1       S002      James   24         2        14.4
2       S003        Mia   21        14        27.3
3       S004       Noah   22        10        23.0
4       S005      Alice   23        14        28.3


In [8]:
df_window = pd.read_csv("student_grade.csv")

# 1. Compute a 3-student rolling moving average of absences
df_window["rolling_avg_absences"] = (
    df_window["absences"].rolling(window=3, min_periods=1).mean()
)

# 2. Compute a cumulative expanding sum of absences from the first row down to the last
df_window["expanding_sum_absences"] = (
    df_window["absences"].expanding(min_periods=1).sum()
)

print("--- WINDOW FUNCTIONS METRICS ---")
print(
    df_window[
        [
            "student_id",
            "first_name",
            "absences",
            "rolling_avg_absences",
            "expanding_sum_absences",
        ]
    ].head(6)
)


--- WINDOW FUNCTIONS METRICS ---
  student_id first_name  absences  rolling_avg_absences  \
0       S001      Alice         4              4.000000   
1       S002      James         2              3.000000   
2       S003        Mia        14              6.666667   
3       S004       Noah        10              8.666667   
4       S005      Alice        14             12.666667   
5       S006      James         1              8.333333   

   expanding_sum_absences  
0                     4.0  
1                     6.0  
2                    20.0  
3                    30.0  
4                    44.0  
5                    45.0  


In [10]:
# Stream 'students_new.csv' by reading it in tiny increments of 15 rows at a time
chunk_iterator = pd.read_csv("student_grade.csv", chunksize=15)

chunk_results = []
print("--- STREAMING AND AGGREGATING DATA BY CHUNKS ---")

# Memory footprint stays minimal as only 15 rows reside in active RAM at any time
for i, chunk in enumerate(chunk_iterator):
    # Perform a quick localized calculation on the current batch
    chunk_summary = chunk.groupby("major")["absences"].mean().reset_index()
    chunk_results.append(chunk_summary)
    print(f"Processed Chunk #{i+1} successfully.")

# Stack all intermediate batch summaries together and compute the absolute final metric
df_final_large = pd.concat(chunk_results, ignore_index=True)
df_final_output = df_final_large.groupby("major")["absences"].mean().reset_index()

print("\n--- FINAL GLOBAL METRICS COMBINED FROM CHUNKS ---")
print(df_final_output)


--- STREAMING AND AGGREGATING DATA BY CHUNKS ---
Processed Chunk #1 successfully.
Processed Chunk #2 successfully.
Processed Chunk #3 successfully.
Processed Chunk #4 successfully.
Processed Chunk #5 successfully.
Processed Chunk #6 successfully.
Processed Chunk #7 successfully.

--- FINAL GLOBAL METRICS COMBINED FROM CHUNKS ---
                  major   absences
0                    AI   7.378571
1      Computer Science  12.392857
2        Cyber Security  10.423810
3          Data Science  13.722222
4  Software Engineering   9.864286
